# Probe Alpha Type of Class 1 in Public Private
In this notebook, we probe the proportion of Alpha type of Class 1 in Public Test and Private Test. There are 3 types of Alpha Class 1 targets listed in the `greeks.csv` file. The types are `Alpha = B, D, G`. From previous probing, we know that public test has 150 samples with 26 positive Class 1 samples. The indices of these samples are listed below. From previous (post comp) probing, we know that private test has 210 samples with 37 positive Class 1 samples. The indices of these samples are listed below. We would know like to know what is the proportion are Alpha=B, D, G? The answer is...

![](https://raw.githubusercontent.com/cdeotte/Kaggle_Images/main/Aug-2023/class1.png)

# Methodology Explained
Version 2 of this notebook verifies that we have the correct 26 test indices to identify the 26 `target=1` Class 1 samples in Public test and correct 37 test indices to identify the 37 `target=1` Class 1 samples in Private test. Version 2 scores `Public LB=0` and `Private LB=0`.

To determine the proportion of `Alpha = B,D,G`, we train a CatBoost classifier to predict the Class 1 type. Then we infer only the 26 Class 1 samples in Public test and 37 Class 1 samples in Private test. We manually override the `Alpha = A` prediction to be `-1` and force the classifier to choose 1 of the other 3 types. 

Locally we validate our CatBoost classifier on 50 repeated 5-Fold validation. From the confusion matrix in Version 5 of this notebook, we see that the true amount of `Alpha = B, D, G` is `0.954, 1.65, and 0.846` ratio of the predicted OOF. Therefore we multiply our predicted quantities by these 3 ratios.

Version 1 of this notebook returns the value of `Alpha = B` for pubic and private. Version 3 returns `Alpha = D` and Version 4 return `Alpha = G`. Version 5 computes the validation confusion matrix.

The Version 1 `Public LB = 9.96310` which is `15 = 9.9631 / (-1*np.log(1e-15)/26/2)`. Therefore classifier predicts 15 `Alpha=B` public Class 1. And we know from local validation that true should be `0.954` ratio of this. Therefore public test has proportion `0.550 = 0.954 * 15 / 26` i.e. 55% `Alpha = B`.

The Version 1 `Private LB = 7.46784` which is `16 = 7.46784 / (-1*np.log(1e-15)/37/2)`. Therefore classifier predicts 16 `Alpha=B` private Class 1. And we know from local validation that true should be `0.954` ratio of this. Therefore private test has proportion `0.412 = 0.954 * 16 / 37` i.e. 41% `Alpha = B`.

We use similar computation to compute `Alpha = D and G` in notebook Versions 3 and 4. More info in the last notebook section called `Compute Results`.

# Load Data
Data load code from https://www.kaggle.com/code/vitalykudelya/gold-medal-top-4-8-18-lines-of-code-late-sub

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from catboost import CatBoostClassifier

train = pd.read_csv('/kaggle/input/icr-identify-age-related-conditions/train.csv')
test = pd.read_csv('/kaggle/input/icr-identify-age-related-conditions/test.csv')
test = test.sort_values('Id')
sample = pd.read_csv('/kaggle/input/icr-identify-age-related-conditions/sample_submission.csv')
greeks = pd.read_csv('/kaggle/input/icr-identify-age-related-conditions/greeks.csv')
greeks['epsilon_date'] = pd.to_datetime(greeks['Epsilon'].replace('Unknown', None), format='%m/%d/%Y')

# mega feature engineering )
train.EJ = train.EJ.eq('B').astype('int')
test.EJ = test.EJ.eq('B').astype('int')

# just use all features
features = [i for i in train.columns if i not in ['Id', 'Class']]

# drop null and old data
train = train.merge(greeks, on='Id')

In [ ]:
print('Train size:',train.shape)
train.head()

# Plot Alpha Type of Class 1 Proportions

In [ ]:
# COMPUTE PROPORTIONS OF CLASS 1 TYPES IN TRAIN DATA
proportions = train.loc[train.Alpha!='A','Alpha'].value_counts(normalize=True).sort_index()

plt.figure(figsize=(20,10))
plt.subplot(1,3,1)
plt.pie(proportions.values, labels=['B','D','G'], autopct='%1.1f%%',
       textprops={'fontsize': 14})
plt.title('Distribution of Class 1 Types\nin TRAIN Data',size=12)

plt.subplot(1,3,2)
plt.pie((0.550,0.127,0.293), labels=['B','D','G'], autopct='%1.1f%%',
       textprops={'fontsize': 14})
plt.title('Distribution of Class 1 Types\nin PUBLIC Test',size=12)

plt.subplot(1,3,3)
plt.pie((0.412,0.312,00.320), labels=['B','D','G'], autopct='%1.1f%%',
       textprops={'fontsize': 14})
plt.title('Distribution of Class 1 Types\nin PRIVATE Test',size=12)

plt.show()

train.Alpha = train.Alpha.map({'A':0, 'B':1, 'D':2, 'G':3})

# Validate CatBoost Model

In [ ]:
%%time

VALIDATE = False
if VALIDATE:
    all_true = []
    all_oof = []

    # REPEATED 5-FOLD VALIDATION
    for fold in range(50):
        np.random.seed(fold)
        valid_idx = np.random.choice(np.arange(len(train)),120,replace=False)
        train_idx = np.setdiff1d(np.arange(len(train)),valid_idx)
        print('#'*25)
        cts = train.loc[valid_idx,'Alpha'].value_counts().values
        print(f'### Fold {fold+1} has valid count A={cts[0]}, B={cts[1]}, D={cts[2]}, G={cts[3]}')

        model = CatBoostClassifier(random_seed=42, verbose=1000, iterations=1000) #task_type="GPU"
        model.fit(train.loc[train_idx,features], train.loc[train_idx,'Alpha'])

        oof = model.predict_proba(train.loc[valid_idx,features])
        true = train.loc[valid_idx,'Alpha'].values
        idx = np.where(true!=0)[0].astype('int')
        oof[(idx,np.zeros(len(idx),dtype='int'))] = -1
        idx = np.where(true==0)[0].astype('int')
        oof[(idx,np.zeros(len(idx),dtype='int'))] = 2
        oof = np.argmax(oof,axis=1)

        all_oof.append(oof)
        all_true.append(true)

        #break

In [ ]:
if VALIDATE:
    true = np.concatenate(all_true,axis=0)
    oof = np.concatenate(all_oof,axis=0)

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
if VALIDATE:
    cm = confusion_matrix(true,oof,labels=[0,1,2,3])
    #disp = ConfusionMatrixDisplay(cm, display_labels=['A','B','D','G'])
    #disp.plot()
    #plt.show()
    print( cm )

    # CONFUSION FOR ITERATIONS=1000 AND 50 REPEAT 5-FOLD:
    #[[4992    0    0    0]
    # [   0  535   18    9]
    # [   0   38   78   67]
    # [   0   13   15  235]]

# Train and Infer Test

In [ ]:
model = CatBoostClassifier(random_seed=42, verbose=100, iterations=1000) 
model.fit(train[features], train['Alpha'])
p = model.predict_proba(test[features])

# Return Answer via LB Score
See notebook versions 1,3,4 for the returned answer of `Alpha=B,D,G` respectively

In [ ]:
PUBLIC1 = np.array([
        14, 26, 32, 42, 71, 101, 103, 105, 106, 170, 173, 176, 179, 187, 
        194, 210, 238, 246, 258, 264, 270, 302, 329, 338, 355, 359])
PUBLIC0 = np.array([  
         1,   2,   6,   9,  10,  12,  15,  16,  20,  23,  24,  27,  28,
        34,  37,  39,  44,  45,  46,  50,  53,  55,  57,  58,  66,  69,
        72,  76,  79,  83,  85,  86,  87,  90,  91,  92,  95,  96, 100,
       107, 109, 111, 112, 113, 114, 123, 125, 126, 129, 132, 133, 134,
       135, 145, 148, 150, 153, 156, 157, 158, 163, 164, 166, 174, 177,
       178, 181, 182, 183, 185, 188, 190, 192, 195, 196, 197, 208, 211,
       214, 217, 219, 221, 222, 224, 225, 227, 229, 232, 234, 236, 237,
       239, 240, 241, 245, 249, 254, 257, 259, 260, 266, 269, 280, 282,
       286, 289, 295, 299, 306, 307, 309, 315, 321, 323, 326, 327, 328,
       330, 333, 343, 344, 346, 351, 352])
PRIVATE1 = np.array([
        13, 21, 35, 36, 54, 73, 102, 108, 121, 124, 130, 136, 143, 147, 
       151, 189, 203, 215, 223, 256, 261, 265, 267, 277, 281, 287, 297, 
       300, 301, 310, 316, 317, 342, 345, 353, 354, 356])

# PREVENT ERROR DURING COMMIT. WILL WORK DURING SUBMIT
PUBLIC1 = PUBLIC1[PUBLIC1<len(test)]
PRIVATE1 = PRIVATE1[PRIVATE1<len(test)]

In [ ]:
p[(PUBLIC1,np.zeros(len(PUBLIC1),dtype='int'))] = -1
p[(PRIVATE1,np.zeros(len(PRIVATE1),dtype='int'))] = -1
pred = np.argmax(p,axis=1)

In [ ]:
# COUNT ALPHA=A in KNOWN CLASS 1 
CLASS = 0
public_ct = (pred[PUBLIC1]==CLASS).sum()
private_ct = (pred[PRIVATE1]==CLASS).sum()

In [ ]:
sub = test[['Id']].copy()

pred0 = np.zeros(len(sub))
pred0[PUBLIC1[public_ct:]] = 1
pred0[PRIVATE1[private_ct:]] = 1

sub['class_0'] = 1-pred0
sub['class_1'] = pred0

In [ ]:
sub.to_csv('submission.csv',index=False)
sub.head()

# Compute Results
Methodology is explained in notebook intro. First we use local validation confusion matrix (from notebook version 5) to determine the ratio of true predictions divided by oof predictions. For `Alpha = B,D,G` this is `0.954, 1.65, 0.846` respectively. Next we use LB scores from notebook versions 1, 3, 4 to compute predicted number of `Alpha = B,D,G` and multiply by the validation ratios.

To return the number of Alpha=B in the LB score of version 2, we first set all 360 predictions to their correct values (LB=0). Then we purposely make a number of `target=1` errors equal to the number of predicted `Alpha=B`. Each `target=1` error increases the public LB score by `-1*np.log(1e-15)/26/2` and increases private LB score by `-1*np.log(1e-15)/37/2` where 26 and 37 are the known number of Class 1 samples in each. These formulas are a result of the balanced log loss metric with eps clipping at `1e-15`.

In [ ]:
B = (535+18+9)/(538+38+13)
D = (38+78+67)/(18+78+15)
G = (13+15+235)/(9+67+235)
B, D, G

In [ ]:
# PUBLIC HAS 15 B
print( 9.9631 / (-1*np.log(1e-15)/26/2) )
# PUBLIC HAS 2 D
print( 1.32841 / (-1*np.log(1e-15)/26/2) )
# PUBLIC HAS 9 G
print( 5.97786 / (-1*np.log(1e-15)/26/2) )

In [ ]:
print('Public Test Alpha=B,D,G is...')
B*15/26, D*2/26, G*9/26

In [ ]:
# PRIVATE HAS 16 B
print( 7.46784 / (-1*np.log(1e-15)/37/2) )
# PRIVATE HAS 7 D
print( 3.26718 / (-1*np.log(1e-15)/37/2) )
# PRIVATE HAS 14 G
print( 6.53436 / (-1*np.log(1e-15)/37/2) )

In [ ]:
print('Private Test Alpha=B,D,G is...')
B*16/37, D*7/37, G*14/37